# Solver Comparison (NetworkX + Pathfinding + Grid)

This notebook generates a single sample at a chosen resolution using `passage.generate.generate_sample`,
then compares NetworkX, Pathfinding and internal Grid (NumPy) solvers from `passage.generate.find_path`.

It records computation time, summarizes results, and visualizes all paths on the elevation map.

We report solve time statistics; for a set of times $t_i$, the mean is $\mu = \frac{1}{N}\sum_i t_i$ and the maximum is $\max_i t_i$.


In [ ]:
import json
import logging
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import pandas as pd
import yaml
from loguru import logger as loguru_logger

# Configure info logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

# Configure loguru for info level
loguru_logger.remove()
loguru_logger.add(sys.stderr, level="INFO")

# Add src to path for imports
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
from passage.generate import generate_sample, find_path, lat_to_str, lon_to_str, _resolve_obstacle_config
from passage.visualization import tensor_to_image

In [ ]:
# Parameters (papermill)
RESOLUTION = 512

In [4]:
# Solvers list (moved to top for visibility)
CONFIG_PATH = ROOT / "config" / "config.yaml"
DEFAULT_COST_MODEL = "elevation"

try:
    with open(CONFIG_PATH) as f:
        _config = yaml.safe_load(f) or {}
    _solver_cfg = _config.get("solver", {})
    _obstacle_settings = _resolve_obstacle_config(_config)
    COST_MODEL = (
        _solver_cfg.get("grid", {}).get("cost_model")
        or _solver_cfg.get("networkx", {}).get("cost_model")
        or _solver_cfg.get("pathfinding", {}).get("cost_model")
        or DEFAULT_COST_MODEL
    )
    OBSTACLES_ENABLED = bool(_obstacle_settings.get("enabled", False))
    OBSTACLE_PENALTY = float(_obstacle_settings.get("obstacle_penalty", 1e8))
except Exception:
    COST_MODEL = DEFAULT_COST_MODEL
    OBSTACLES_ENABLED = False
    OBSTACLE_PENALTY = 1e8

SOLVERS = [
    # NetworkX
    {"name": "networkx:astar", "heuristic": "euclidean", "cost_model": COST_MODEL, "label": "NetworkX A* (Euclidean)"},
    {
        "name": "networkx:astar_bi",
        "heuristic": "euclidean",
        "cost_model": COST_MODEL,
        "label": "NetworkX A* (Bidirectional)",
    },
    {"name": "networkx:dijkstra", "cost_model": COST_MODEL, "label": "NetworkX Dijkstra"},
    {"name": "networkx:dijkstra_bi", "cost_model": COST_MODEL, "label": "NetworkX Dijkstra (Bidirectional)"},
    {"name": "networkx:bellman_ford", "cost_model": COST_MODEL, "label": "NetworkX Bellman-Ford"},
    {"name": "networkx:unweighted", "cost_model": COST_MODEL, "label": "NetworkX Unweighted (BFS)"},
    # Pathfinding
    {
        "name": "pathfinding:astar",
        "heuristic": "euclidean",
        "cost_model": COST_MODEL,
        "label": "Pathfinding A* (Euclidean)",
    },
    {"name": "pathfinding:dijkstra", "cost_model": COST_MODEL, "label": "Pathfinding Dijkstra"},
    {
        "name": "pathfinding:best_first",
        "heuristic": "euclidean",
        "cost_model": COST_MODEL,
        "label": "Pathfinding Best-First",
    },
    # Grid (NumPy-based internal solvers)
    {"name": "grid:astar", "heuristic": "euclidean", "cost_model": COST_MODEL, "label": "Grid A* (NumPy)"},
    {"name": "grid:dijkstra", "cost_model": COST_MODEL, "label": "Grid Dijkstra (NumPy)"},
]

## Configuration

Choose a resolution and where to store the generated sample.

In [5]:
# Sample generation settings
RESOLUTION = 256  # 64, 128, 256, 512, 1024, 2048, 4096
SAMPLE_INDEX = 0
CENTER_LAT = None  # Set to int if known
CENTER_LON = None  # Set to int if known
DISABLE_IMAGES = False
OVERRIDE_EXISTING = True

# Paths
OUTPUTS_DIR = ROOT / "outputs" / "solve"
TILES_DIR = ROOT / "outputs" / "download" / "tiles"
CALIBRATE_PATH = ROOT / "outputs" / "calibrate" / "calibrate.json"
OBSTACLE_SETTINGS = _resolve_obstacle_config(_config if "_config" in globals() else {})
USE_OBSTACLES = bool(OBSTACLE_SETTINGS.get("enabled", False))
OBSTACLE_PENALTY = float(OBSTACLE_SETTINGS.get("obstacle_penalty", 1e8))

# Solver defaults for sample generation (used only to ensure a valid path exists)
GEN_SOLVER_CONFIG = {
    "name": "grid:astar",
    "diagonal_movement": "always",
    "weight": 30.0,
    "cost_model": COST_MODEL,
    "heuristic": "euclidean",
}

# Solver comparison settings
WEIGHT = 30.0
DIAGONAL_MOVEMENT = "always"
COST_MODEL = COST_MODEL  # Pathfinding supports elevation, distance, absolute_elevation

## Generate or Load Sample

This uses `generate_sample` from `passage.generate` to create a single tensor if it does not exist yet.

In [6]:
# Validate required inputs
if not CALIBRATE_PATH.exists():
    raise FileNotFoundError(f"Calibration file not found: {CALIBRATE_PATH}. Run 'passage calibrate' first.")
if not TILES_DIR.exists():
    raise FileNotFoundError(f"Tiles directory not found: {TILES_DIR}. Run 'passage download' first.")

# Load calibration data
with open(CALIBRATE_PATH) as f:
    calibration = json.load(f)
global_min = float(calibration["global_min"])
global_max = float(calibration["global_max"])
tile_stats_df = pd.DataFrame(calibration["tile_stats"])


def resolve_sample_paths(outputs_dir: Path, resolution: int, index: int, center_lat=None, center_lon=None):
    tensors_root = outputs_dir / str(resolution) / "tensors"
    metadata_root = outputs_dir / str(resolution) / "metadata"
    if center_lat is not None and center_lon is not None:
        lat_dir = lat_to_str(int(center_lat))
        lon_dir = lon_to_str(int(center_lon))
        tensor_path = tensors_root / lat_dir / lon_dir / f"{index:010d}.npy"
        metadata_path = metadata_root / lat_dir / lon_dir / f"{index:010d}.yaml"
        return tensor_path, metadata_path
    tensor_matches = list(tensors_root.rglob(f"{index:010d}.npy"))
    metadata_matches = list(metadata_root.rglob(f"{index:010d}.yaml"))
    tensor_path = tensor_matches[0] if tensor_matches else tensors_root / f"{index:010d}.npy"
    metadata_path = metadata_matches[0] if metadata_matches else metadata_root / f"{index:010d}.yaml"
    return tensor_path, metadata_path


# Generate sample if not present (or if OVERRIDE_EXISTING is True)
tensor_path, metadata_path = resolve_sample_paths(OUTPUTS_DIR, RESOLUTION, SAMPLE_INDEX, CENTER_LAT, CENTER_LON)

if OVERRIDE_EXISTING or not tensor_path.exists() or not metadata_path.exists():
    OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
    args = (
        SAMPLE_INDEX,
        TILES_DIR,
        OUTPUTS_DIR,
        RESOLUTION,
        tile_stats_df,
        global_min,
        global_max,
        DISABLE_IMAGES,
        GEN_SOLVER_CONFIG,
        OBSTACLE_SETTINGS,
    )
    index, success, message, duration = generate_sample(args)
    print(f"Generate sample: success={success}, message='{message}', duration={duration:.2f}s")
    if not success:
        raise RuntimeError("Sample generation failed. Try a different resolution or re-run calibration.")
else:
    print(f"Using existing tensor: {tensor_path}")

# Load tensor (shape: H, W, 3) -> channels: elevation (normalized), markers, obstacles
tensor = np.load(tensor_path)
elevation_norm = tensor[:, :, 0]
markers = tensor[:, :, 1]
obstacles_mask = tensor[:, :, 2] if tensor.shape[2] > 2 else np.zeros_like(markers)
# elevation_raw is the unnormalized elevation in the original units
elevation_raw = elevation_norm * (global_max - global_min) + global_min

2026-01-19 22:04:35.292 | INFO     | passage.generate:generate_sample:1243 - Sample 0: success=True, elevation_attempts=1, marker_retries=1, path_length_no_obstacles=198, path_length_with_obstacles=198, grid_shape=(10800,10800), duration=2.61s


Generate sample: success=True, message='Generated sample 0', duration=2.62s


## Extract Markers

Use the marker channel to recover the start/end coordinates.

In [7]:
start_positions = np.argwhere(np.isclose(markers, 0.0))
end_positions = np.argwhere(markers > 0.5)
if len(start_positions) < 1 or len(end_positions) < 1:
    raise ValueError("Expected start and end markers in the tensor.")

start = tuple(start_positions[0])
end = tuple(end_positions[0])

print(f"Start: {start}")
print(f"End:   {end}")
print(f"Euclidean distance: {np.hypot(start[0] - end[0], start[1] - end[1]):.1f} pixels")

Start: (np.int64(102), np.int64(216))
End:   (np.int64(55), np.int64(115))
Euclidean distance: 111.4 pixels


## Visualize Elevation Map

Visualize elevation map and markers.

In [8]:
# Display elevation map with matplotlib and show markers if available
fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(elevation_norm, cmap="terrain", origin="upper")
ax.set_title(f"Elevation (normalized) - {RESOLUTION}x{RESOLUTION} (obstacles={'on' if USE_OBSTACLES else 'off'})")
plt.colorbar(im, ax=ax, label="Normalized elevation")

if USE_OBSTACLES:
    obs_overlay = np.ma.masked_where(obstacles_mask == 0, obstacles_mask)
    ax.imshow(obs_overlay, cmap="Reds", alpha=0.35, origin="upper")

# Plot start/end markers if present
try:
    s = start
    e = end
except NameError:
    start_positions = np.argwhere(np.isclose(markers, 0.0))
    end_positions = np.argwhere(markers > 0.5)
    s = tuple(start_positions[0]) if len(start_positions) > 0 else None
    e = tuple(end_positions[0]) if len(end_positions) > 0 else None

if s is not None:
    ax.plot(s[1], s[0], "o", color="lime", markersize=8, label="Start")
if e is not None:
    ax.plot(e[1], e[0], "x", color="cyan", markersize=8, label="End")

ax.legend()
ax.axis("off")
plt.tight_layout()
plt.show()

# Show tensor_to_image visualization (terrain colormap + overlays)
# Extract path pixels and elevation bounds from metadata, then call tensor_to_image
path_pixels = None
elev_range = None
try:
    if metadata_path.exists():
        with open(metadata_path) as f:
            md_local = yaml.safe_load(f)
        if md_local is not None:
            # Prefer with_obstacles when obstacles enabled, else without_obstacles
            if "paths" in md_local:
                path_section = md_local.get("paths", {}).get("with_obstacles") or md_local.get("paths", {}).get(
                    "without_obstacles"
                )
                if path_section is None:
                    path_section = md_local.get("path", {})
            else:
                path_section = md_local.get("path", {})

            if path_section:
                path_pixels = [(p["i"], p["j"]) for p in path_section.get("pixels", [])]

            crop_info = md_local.get("crop", {})
            if "min_elevation" in crop_info and "max_elevation" in crop_info:
                elev_range = (crop_info.get("min_elevation"), crop_info.get("max_elevation"))
except Exception:
    path_pixels = None

tensor_vis = np.stack([elevation_norm, markers, obstacles_mask], axis=-1)
img_bgr = tensor_to_image(tensor_vis, path_pixels=path_pixels, elev_range=elev_range)
img_rgb = img_bgr[:, :, ::-1]

fig2, ax2 = plt.subplots(figsize=(8, 8))
ax2.imshow(img_rgb, origin="upper")
ax2.set_title("tensor_to_image visualization (terrain + obstacles + markers)")
ax2.axis("off")
plt.tight_layout()
plt.show()

## Run Solver Comparison

Compare all NetworkX and Pathfinding solvers and record execution time.

In [9]:
# Apply obstacles if configured
elevation_input = elevation_raw
if USE_OBSTACLES:
    elevation_input = elevation_raw.copy()
    elevation_input[obstacles_mask == 1] = elevation_input.max() + OBSTACLE_PENALTY

# Prebuild NetworkX graphs for each cost_model used by networkx solvers (reuse if multiple solvers use same model)
networkx_solvers = [s for s in SOLVERS if s["name"].split(":", 1)[0] == "networkx"]
G_prebuilt_map = {}
if networkx_solvers:
    from passage.generate import build_graph_from_elevation

    # Build a shared graph per cost_model (defaults to global COST_MODEL)
    cost_models = set(s.get("cost_model", COST_MODEL) for s in networkx_solvers)
    for cost_model_name in cost_models:
        G_prebuilt_map[cost_model_name] = build_graph_from_elevation(
            elevation_input, weight=WEIGHT, diagonal_movement=DIAGONAL_MOVEMENT, cost_model=cost_model_name
        )

results = []

for solver_info in SOLVERS:
    solver_config = {
        "name": solver_info["name"],
        "diagonal_movement": DIAGONAL_MOVEMENT,
        "weight": WEIGHT,
        "cost_model": solver_info.get("cost_model", COST_MODEL),
    }
    if "heuristic" in solver_info:
        solver_config["heuristic"] = solver_info["heuristic"]

    t0 = time.perf_counter()

    backend_name = solver_info["name"].split(":", 1)[0]
    if backend_name == "networkx":
        prebuilt = G_prebuilt_map.get(solver_config["cost_model"])
        path = find_path(elevation_input, start, end, solver_config, prebuilt_graph=prebuilt)
    else:
        path = find_path(elevation_input, start, end, solver_config)

    elapsed = (time.perf_counter() - t0) * 1000

    results.append(
        {
            "solver": solver_info["label"],
            "backend": backend_name,
            "path_length": len(path),
            "time_ms": elapsed,
            "path": path,
        }
    )

    print(f"{solver_info['label']}: path_len={len(path)}, time={elapsed:.1f} ms")


timings_df = pd.DataFrame(
    [
        {"Solver": r["solver"], "Backend": r["backend"], "Path Length": r["path_length"], "Time (ms)": r["time_ms"]}
        for r in results
    ]
)
csv_path = OUTPUTS_DIR / f"solver_times_{RESOLUTION}_{SAMPLE_INDEX:010d}.csv"
csv_path.parent.mkdir(parents=True, exist_ok=True)

timings_df.to_csv(csv_path, index=False)
print(f"Saved timings to: {csv_path}")

NetworkX A* (Euclidean): path_len=109, time=48.6 ms
NetworkX A* (Bidirectional): path_len=109, time=45.6 ms
NetworkX Dijkstra: path_len=109, time=250.8 ms
NetworkX Dijkstra (Bidirectional): path_len=109, time=108.1 ms


2026-01-19 22:04:40.237 | WARNING  | passage.generate:find_path:234 - Pathfinding backend does not support directional cost models; falling back to networkx


NetworkX Bellman-Ford: path_len=109, time=1220.6 ms
NetworkX Unweighted (BFS): path_len=102, time=68.0 ms


2026-01-19 22:04:41.439 | WARNING  | passage.generate:find_path:234 - Pathfinding backend does not support directional cost models; falling back to networkx


Pathfinding A* (Euclidean): path_len=109, time=1202.5 ms


2026-01-19 22:04:42.683 | WARNING  | passage.generate:find_path:234 - Pathfinding backend does not support directional cost models; falling back to networkx
2026-01-19 22:04:42.684 | WARNING  | passage.generate:find_path:255 - Unknown networkx solver 'best_first', falling back to astar


Pathfinding Dijkstra: path_len=109, time=1243.8 ms
Pathfinding Best-First: path_len=109, time=1198.2 ms
Grid A* (NumPy): path_len=109, time=27.3 ms
Grid Dijkstra (NumPy): path_len=109, time=58.9 ms
Saved timings to: ~/projects/PASSAGE/passage/outputs/solve/solver_times_256_0000000000.csv


**Note:** The *Grid* solvers (`grid:astar`, `grid:dijkstra`) are internal NumPy‑based implementations that avoid constructing a NetworkX graph. They typically run faster and use less memory for large grids (e.g., 1024, 2048, 4096). Consider using these when performance is a priority.

## Results Summary

Sort the solvers by execution time.

In [10]:
df = timings_df.sort_values("Time (ms)").reset_index(drop=True)
df

,Solver,Backend,Path Length,Time (ms)
0,Grid A* (NumPy),grid,109,27.321724
1,NetworkX A* (Bidirectional),networkx,109,45.593675
2,NetworkX A* (Euclidean),networkx,109,48.645599
3,Grid Dijkstra (NumPy),grid,109,58.860766
4,NetworkX Unweighted (BFS),networkx,102,68.025358
5,NetworkX Dijkstra (Bidirectional),networkx,109,108.109485
6,NetworkX Dijkstra,networkx,109,250.817653
7,Pathfinding Best-First,pathfinding,109,1198.154332
8,Pathfinding A* (Euclidean),pathfinding,109,1202.489076
9,NetworkX Bellman-Ford,networkx,109,1220.630193


## Timing Bar Chart

In [11]:
fig, ax = plt.subplots(figsize=(10, 6))
y_pos = range(len(df))
bars = ax.barh(y_pos, df["Time (ms)"], color="steelblue")
ax.set_yticks(y_pos)
ax.set_yticklabels(df["Solver"])
ax.set_xlabel("Time (ms)")
ax.set_title(f"Solver Comparison - {RESOLUTION}x{RESOLUTION}")
ax.invert_yaxis()

# Add time labels
for i, (bar, time_val) in enumerate(zip(bars, df["Time (ms)"])):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2, f"{time_val:.1f} ms", va="center", fontsize=9)

plt.tight_layout()
plt.show()

## Path Visualization (All Solvers)

Overlay all solver paths on the elevation map.

In [15]:
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(elevation_norm, cmap="terrain", alpha=0.85, origin="upper")

if USE_OBSTACLES:
    obs_overlay = np.ma.masked_where(obstacles_mask == 0, obstacles_mask)
    ax.imshow(obs_overlay, cmap="Reds", alpha=0.25, origin="upper")

# Use a colormap for different solvers

colors = cm.tab20(np.linspace(0, 1, len(results)))

for idx, result in enumerate(results):
    path = result["path"]
    if len(path) == 0:
        continue
    path_arr = np.array(path)
    ax.plot(path_arr[:, 1], path_arr[:, 0], linewidth=2, color=colors[idx], label=result["solver"])

ax.plot(start[1], start[0], "o", color="lime", markersize=8, label="Start")
ax.plot(end[1], end[0], "x", color="cyan", markersize=8, label="End")

ax.set_title("All Solver Paths")
ax.legend(loc="upper left", fontsize=8, bbox_to_anchor=(1.02, 1))
ax.axis("off")
plt.tight_layout()
plt.show()

## Path Visualization (Per Solver)

Show one subplot per solver to compare route shapes.

In [16]:
n_solvers = len(results)
cols = min(3, n_solvers)
rows = (n_solvers + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
if rows == 1:
    axes = axes.reshape(1, -1)
if cols == 1:
    axes = axes.reshape(-1, 1)

for idx, result in enumerate(results):
    r = idx // cols
    c = idx % cols
    ax = axes[r, c]

    ax.imshow(elevation_norm, cmap="terrain", alpha=0.85, origin="upper")

    if USE_OBSTACLES:
        obs_overlay = np.ma.masked_where(obstacles_mask == 0, obstacles_mask)
        ax.imshow(obs_overlay, cmap="Reds", alpha=0.25, origin="upper")

    path = result["path"]
    if len(path) > 0:
        path_arr = np.array(path)
        ax.plot(path_arr[:, 1], path_arr[:, 0], "r-", linewidth=2)

    ax.plot(start[1], start[0], "o", color="lime", markersize=6)
    ax.plot(end[1], end[0], "x", color="cyan", markersize=6)
    ax.set_title(f"{result['solver']}\n{result['time_ms']:.1f}ms, {result['path_length']} px", fontsize=9)
    ax.axis("off")

# Hide empty subplots
for idx in range(n_solvers, rows * cols):
    r = idx // cols
    c = idx % cols
    axes[r, c].axis("off")

plt.tight_layout()
plt.show()